# LLM From Scratch + Post-Training + Agentic RAG — Colab training
Runtime → Change runtime type → **T4 GPU**. Run cells top to bottom.

Add your Hugging Face write token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar). Checkpoints sync to the Hub, so a disconnect only costs the steps since the last save.

In [ ]:
import os
# New Colab session -> clone. Same session -> just pull the latest fixes.
if os.path.exists('/content/llm-from-scratch-agentic-rag'):
    %cd /content/llm-from-scratch-agentic-rag
    !git pull -q
else:
    !git clone -q https://github.com/ahmedashrafk13/llm-from-scratch-agentic-rag.git /content/llm-from-scratch-agentic-rag
    %cd /content/llm-from-scratch-agentic-rag
!pip install -q transformers==4.46.3 datasets==3.1.0 peft==0.13.2 bitsandbytes accelerate sentence-transformers==3.3.1 faiss-cpu rank_bm25 regex gradio
from google.colab import userdata
from huggingface_hub import login, whoami
try:
    login(userdata.get('HF_TOKEN'))
except Exception:
    print('HF_TOKEN secret not found - paste your token below')
    login()
HF_USER = whoami()['name']
HUB = f'{HF_USER}/llm-scratch-checkpoints'
print(HUB)

In [ ]:
!python -m pytest tests -q

## 1. Data + tokenizer
Each step saves to Google Drive as soon as it finishes and is **skipped** next time if its output is already there. The tokenizer is trained once, ever.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, subprocess
D = '/content/drive/MyDrive/llm-scratch'
P = 'data/processed'
os.makedirs(f'{D}/processed', exist_ok=True); os.makedirs(P, exist_ok=True)
!cp -rn {D}/processed/. {P}/ 2>/dev/null; cp -n {D}/tokenizer.json tokenizer/ 2>/dev/null; true

def step(name, done_file, cmd):
    if os.path.exists(done_file):
        print(f'[skip] {name} (already done)'); return
    print(f'[run ] {name}')
    subprocess.run(cmd, shell=True, check=True)
    !cp -r {P}/. {D}/processed/
    if os.path.exists('tokenizer/tokenizer.json'): subprocess.run(f'cp tokenizer/tokenizer.json {D}/', shell=True)
    print(f'[save] {name} -> Drive')

step('download + clean',   f'{P}/fineweb.jsonl',           'python data/prepare_pretrain.py')
step('train tokenizer',    'tokenizer/tokenizer.json',     'python tokenizer/train_tokenizer.py')
step('tokenize corpus',    f'{P}/train.bin',               'python data/tokenize_corpus.py')
step('build SFT/RAG data', f'{P}/domain_eval.jsonl',       'python data/build_sft.py')
step('hallucination set',  'evaluation/hallucination_testset.jsonl', 'python evaluation/build_hallucination_set.py')

## 2. Pre-training (~1 h on T4)
If the session disconnects: re-run the setup + data cells, then this one — `--resume` pulls the last checkpoint from the Hub.

In [ ]:
!python training/pretrain.py --batch 32 --accum 1 --resume --hub-repo {HUB}

## 3. SFT → DPO → reward model (~20 min)

In [ ]:
!python training/sft.py
!python training/dpo.py
!python training/reward_model.py
!cp -r checkpoints {D}/

## 4. QLoRA on Qwen2.5-1.5B (~35 min)

In [ ]:
!python training/qlora.py
!cp -r checkpoints {D}/

## 5. Evaluation + report (~1–1.5 h)

In [ ]:
!python evaluation/benchmarks.py
!python evaluation/retrieval_eval.py
!python evaluation/qa_eval.py
!python evaluation/make_report.py
!cp -r results {D}/

## 6. Upload trained models to the Hub (optional)

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(f'{HF_USER}/llm-scratch-37m', exist_ok=True)
api.upload_folder(folder_path='checkpoints', repo_id=f'{HF_USER}/llm-scratch-37m', ignore_patterns=['pretrain/*'])
print('done')